# bt → canonical bundle → implementation-risk audit

The same toy strategy in **bt 1.2.0**: export, canonical bundle 1.0, check
against the independent reference, and the implementation-risk audit on the
correct run and on a run with an injected one-bar look-ahead.

bt prepends a virtual row one calendar day before the data to hold the
initial capital. The adapter maps it as evidence, so the reference carries the
same initial-capital observation (`prestart=`); against an engine without such
a row it would show up as one-sided evidence. Outputs below come from
`jupyter nbconvert --execute --to notebook --inplace 02_bt.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd

import audit_kit as kit
import backtest_auditor
import toy_strategy as ts
import bt
from backtest_auditor.implementation_risk import (
    ComparisonLevel,
    build_v1_5d2_service,
    export_bt_result,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)
REPO = Path.cwd().resolve().parents[1]
package_dir = Path(backtest_auditor.__file__).resolve().parent
print(
    "backtest_auditor",
    backtest_auditor.__version__,
    "from",
    package_dir.relative_to(REPO),
)
print("bt", bt.__version__)

backtest_auditor 1.6.0 from src/backtest_auditor
bt 1.2.0


## 1. The data and the independent reference

`toy_strategy.py` holds the bars and computes the expected result with pandas
and `decimal` only (no engine, no auditor): ten fills, five round trips
including one short, final equity USD 99,844.00.

In [2]:
prices = ts.load_prices()
expected = ts.expected_run()
print(len(prices), "bars,", prices.index[0], "to", prices.index[-1])
prices.head(3)

400 bars, 2024-01-02 21:00:00+00:00 to 2025-07-14 20:00:00+00:00


,open,high,low,close,volume
timestamp,,,,,
2024-01-02 21:00:00+00:00,100.00,100.23,99.79,100.00,1000000.0
2024-01-03 21:00:00+00:00,100.19,101.07,100.06,100.99,1000000.0
2024-01-04 21:00:00+00:00,100.69,101.40,97.34,97.43,1000000.0


## 2. The strategy in bt

A bt `Algo` computes both averages from the data bt has revealed so far and
transacts the share difference at bt's price for the day, the close.
Commission is bt's `commissions` callable.

In [3]:
class MovingAverageCross(bt.Algo):
    """Holds 100 x signal shares; the signal is lagged `lag` bars."""

    def __init__(self, lag: int) -> None:
        super().__init__()
        self.lag = lag

    def __call__(self, target) -> bool:
        closes = target.universe[ts.INSTRUMENT].dropna()
        if len(closes) <= self.lag:
            return True
        seen = closes.iloc[: len(closes) - self.lag]
        fast = seen.rolling(ts.FAST).mean().iloc[-1]
        slow = seen.rolling(ts.SLOW).mean().iloc[-1]
        signal = 1 if fast > slow else 0
        session = (
            seen.index[-1]
            .tz_convert(ts.EXCHANGE_TIMEZONE)
            .normalize()
            .tz_localize(None)
        )
        if ts.SHORT_FIRST_SIGNAL_SESSION <= session <= ts.SHORT_LAST_SIGNAL_SESSION:
            signal = -1
        wanted = ts.QUANTITY * signal
        held = (
            target.children[ts.INSTRUMENT].position
            if ts.INSTRUMENT in target.children
            else 0.0
        )
        if wanted != held:
            target.transact(wanted - held, child=ts.INSTRUMENT)
        return True


def run_bt(lag: int) -> bt.backtest.Result:
    data = prices[["close"]].rename(columns={"close": ts.INSTRUMENT})
    strategy = bt.Strategy(
        "ma_cross", [MovingAverageCross(lag)], children=[ts.INSTRUMENT]
    )
    backtest = bt.Backtest(
        strategy,
        data,
        initial_capital=float(ts.INITIAL_CASH),
        commissions=lambda quantity, price: float(ts.FEE),
        integer_positions=True,
        progress_bar=False,
    )
    return bt.run(backtest)


result = run_bt(lag=1)
print("final value:", result.backtests["ma_cross"].strategy.values.iloc[-1])

final value:

 99844.0


## 3. Export, map to canonical bundle 1.0, check against the reference

`export_bt_result` reads the completed `bt.Result` and refuses any bt other
than 1.2.0. bt records per-day fees, not per-fill links, so costs are not
linked to fills; there is no trade level.

In [4]:
EXPORTS = kit.OUTPUT / "bt"
exported = export_bt_result(
    result,
    EXPORTS / "export-correct",
    run_id="bt-correct",
    base_currency=ts.CURRENCY,
    overwrite=True,
)
adapted = build_v1_5d2_service().adapt(exported)
print("adapter warnings:", [warning.code for warning in adapted.warnings])
bundle = kit.write_bundle(adapted.artifacts, EXPORTS / "bt-correct.zip")
kit.bundle_summary(EXPORTS / "bt-correct.zip")

adapter warnings: ['bt.aggregate_fees_unlinked', 'bt.transactions_reconstructed_orders']


,file,records,sha256
0,cash_balances.csv,401,2f05898a4ce3…
1,costs.csv,10,7f3d008bcd61…
2,equity.csv,401,5a557766dfec…
3,fills.csv,10,917013169c78…
4,orders.csv,10,f0194e11dfd5…
5,positions.csv,401,f844c2ea6940…
6,returns.csv,400,b62642aeb135…


In [5]:
kit.compare_fills(bundle, expected)

,timestamp (UTC),engine,reference,same
0,2024-03-25 20:00,buy 100 @ 89.51,buy 100 @ 89.51,True
1,2024-06-19 20:00,sell 100 @ 97.33,sell 100 @ 97.33,True
2,2024-08-12 20:00,buy 100 @ 101.88,buy 100 @ 101.88,True
3,2024-09-25 20:00,sell 100 @ 94.88,sell 100 @ 94.88,True
4,2024-11-11 21:00,buy 100 @ 97.56,buy 100 @ 97.56,True
5,2025-01-24 21:00,sell 100 @ 94.96,sell 100 @ 94.96,True
6,2025-02-07 21:00,sell 100 @ 96.59,sell 100 @ 96.59,True
7,2025-03-07 21:00,buy 100 @ 91.56,buy 100 @ 91.56,True
8,2025-04-15 20:00,buy 100 @ 102.53,buy 100 @ 102.53,True
9,2025-06-03 20:00,sell 100 @ 97.82,sell 100 @ 97.82,True


In [6]:
kit.final_value_check(bundle, expected)

,engine,reference
,,
final equity,99844,99844.00


## 4. Implementation-risk audit of the correct run

The reference bundle is built from `toy_strategy.py` alone. Levels are aligned
on economics (time, instrument, side), not on engine-local ids. Materiality
thresholds sit far above float noise and far below the effect of a one-bar
shift (see `audit_kit.MEASURED_FIELDS`). Levels compared for bt:
fill, position, cash, cost, equity, return.

How to read the summary: a score near zero means every measured difference
is float noise below its threshold. No score with uncertainty `not_assessed`
means there was no numeric observation to classify, so read it with the
findings: with no `adverse` or `evidence_gap` finding it is exact agreement on
every measured field; with `evidence_gap` findings it means records could not
be paired at all (the NautilusTrader look-ahead run shows that case).

In [7]:
LEVELS = (
    ComparisonLevel.FILL,
    ComparisonLevel.POSITION,
    ComparisonLevel.CASH,
    ComparisonLevel.COST,
    ComparisonLevel.EQUITY,
    ComparisonLevel.RETURN,
)
reference_bundle = kit.reference_artifacts(
    prestart=prices.index[0] - pd.DateOffset(days=1)
)
correct = kit.implementation_risk(reference_bundle, bundle, LEVELS, name="bt-correct")
kit.summary(correct)

,implementation_risk_score,uncertainty,score_withheld_reason,supporting findings
candidate,,,,
bt-correct,0.00000039142271458825,low,None,209


In [8]:
kit.levels_table(correct)

,status,matched_equal,matched_different
level,,,
fill,compared,10,0
position,compared,0,401
cash,compared,0,401
cost,compared,10,0
equity,compared,0,401
return,compared,192,208


`matched_different` counts matched records in which any field differs,
including fields that only one side fills in (the reference leaves market
values and exposures empty) and labels that differ by design (a trade's
grouping method). Which fields differ, and whether the measurement policy
measures them:

In [9]:
kit.differing_fields(correct)

,level,field,records,measured
0,position,market_price,400,False
1,position,market_value,401,False
2,cash,settled,401,False
3,equity,gross_exposure,401,False
4,equity,net_exposure,401,False
5,return,value,208,True


In [10]:
print(kit.report_excerpt(correct))

# Backtest Implementation-Risk Report

- Report schema: `1.0`
- Reconciliation: `toy-ma-cross`
- Reference: `reference-pandas` (pandas-reference)
- Candidates: 1
- Overall status: not produced

> The report identifies implementation divergence against a declared reference baseline. It does not establish that any engine is correct, prove causation, certify strategy skill, or predict future performance.

## Policy identity

| Policy | SHA-256 |
|---|---|
| Normalization | `82e69b78c0142792b9f720a1fd09af8b7b7fdeef6805f58734ae6cb6f2020fec` |
| Comparison | `0dd57af165d1007d635c4eb6689584ffb24eba5684338b4facfeb695a4754b37` |
| Measurement | `3dbaeab3a8e9f86bb7462b76a77501d53b498bb15cc74f5c4333be63257c4c6c` |
| Assessment | `929228caacbb98e9acaebc6ccc5ea1452f3c09ef1a5f4ceb20f5f5d4f9e2eaa1` |

## Candidate `bt-correct`

- Engine: bt
- Implementation-risk score: 0.00000039142271458825
- Uncertainty: low

### Comparison levels

| Level | Status | Alignment | Exact match | Records |
|---|---|---

## 5. The same strategy with a one-bar look-ahead

The defect: the position follows the signal of the same bar, so the strategy
trades on a close it could not have known. Every fill moves one bar earlier.

In [11]:
exported = export_bt_result(
    run_bt(lag=0),
    EXPORTS / "export-lookahead",
    run_id="bt-lookahead",
    base_currency=ts.CURRENCY,
    overwrite=True,
)
lookahead_bundle = kit.write_bundle(
    build_v1_5d2_service().adapt(exported).artifacts, EXPORTS / "bt-lookahead.zip"
)
lookahead = kit.implementation_risk(
    reference_bundle, lookahead_bundle, LEVELS, name="bt-lookahead"
)
kit.summary(lookahead)

,implementation_risk_score,uncertainty,score_withheld_reason,adverse findings,evidence_gap findings
candidate,,,,,
bt-lookahead,None,high,score withheld because selected divergence evidence is incomplete,907,5


In [12]:
kit.families_table(lookahead)

,status,maximum threshold ratio,score contribution
family,,,
transaction_cost,partial,NaN,NaN
sizing,partial,200,25
accounting,partial,1041300,25
return,measured,224327.5181765377451078575,25


In [13]:
print(kit.report_findings(lookahead, 4))

## Independent findings

| Code | Kind | Candidate | Family | Summary | Limitation | Action |
|---|---|---|---|---|---|---|
| IMPLEMENTATION_FAMILY_EVIDENCE_IMPAIRED | evidence_gap | bt-lookahead | transaction_cost | transaction_cost evidence is partial. | Materiality and scoring are bounded by incomplete evidence. | Resolve unavailable, one-sided, ambiguous, or non-numeric evidence. |
| IMPLEMENTATION_FAMILY_EVIDENCE_IMPAIRED | evidence_gap | bt-lookahead | sizing | sizing evidence is partial. | Materiality and scoring are bounded by incomplete evidence. | Resolve unavailable, one-sided, ambiguous, or non-numeric evidence. |
| IMPLEMENTATION_DIVERGENCE_EXTREME | adverse | bt-lookahead | sizing | sizing quantity divergence is extreme at 200× threshold. | The classification is conditional on the supplied field-specific threshold. | Review the canonical records and the declared materiality threshold. |
| IMPLEMENTATION_DIVERGENCE_EXTREME | adverse | bt-lookahead | sizing | sizing quantit

## What this shows, and what it does not

- The bt run and the independent reference agree on every measured
  field at every level compared above (other differences are confined to
  fields the reference leaves empty, labels that differ by design, or float
  noise below the thresholds), and the audit names no defect.
- The injected one-bar look-ahead produces adverse findings, and the score is withheld because
  evidence on some levels became one-sided.
- The audit compares evidence against a declared reference under explicit
  thresholds. It does not prove either implementation correct, and it says
  nothing about whether the strategy is any good.
- The full JSON and Markdown reports of both runs are in `_output/` after
  executing this notebook.